# 01 — Qwen Processor
Consumes the existing Drive topic list and prepares up to 40 jobs. No image/audio/video generation happens here.

In [ ]:
import os,sys,subprocess
ROOT="/content/black-history-factory"
REPO_URL="https://github.com/jonbBla/black-history-factory.git"  # put your GitHub repo URL here if you want automatic git clone
if not os.path.exists(ROOT):
    if not REPO_URL: raise RuntimeError("Set REPO_URL to your GitHub repository before running.")
    subprocess.run(["git","clone",REPO_URL,ROOT],check=True)
sys.path.insert(0,ROOT)
from factory.drive import mount_drive,DrivePaths
from factory.config import Config
MYDRIVE=mount_drive(); paths=DrivePaths(os.path.join(MYDRIVE,"BLACK_HISTORY_FACTORY")); paths.ensure_tree(); config=Config.load(paths.root)
print(paths.root)


In [ ]:
from factory.qwen_client import QwenClient
from factory import topic_engine,research_engine,fact_checker,visual_bible,script_engine,scene_engine,status
from factory.utils import read_json,write_json_atomic,now_iso
qwen=QwenClient("Qwen/Qwen3-14B-Instruct",load_in_4bit=True)

def ready_count():
    return sum(1 for j in os.listdir(paths("02_JOBS")) if (read_json(paths.manifest(j),{}) or {}).get("status")=="QWEN_READY")

while ready_count()<int(config.prepared_job_target):
    topic,job_id=topic_engine.claim_next_topic(paths,"qwen")
    if not topic: print("No unused topics remain."); break
    try:
        status.set_processor(paths,"qwen","running",job_id,topic.title)
        research=research_engine.run(paths,job_id,topic,qwen)
        verified=fact_checker.run(paths,job_id,research,qwen)
        if verified.get("verdict")=="REJECT":
            d=read_json(paths.manifest(job_id),{}); d.update(status="REJECTED",reason=verified.get("issues",[])); write_json_atomic(paths.manifest(job_id),d); continue
        vb=visual_bible.run(paths,job_id,topic,research,config,qwen)
        narration=script_engine.run(paths,job_id,topic,verified,config,qwen)
        scenes=scene_engine.run(paths,job_id,narration,vb,config,qwen)
        d=read_json(paths.manifest(job_id),{}); d.update(status="QWEN_READY",scene_count=len(scenes),prepared_at=now_iso()); write_json_atomic(paths.manifest(job_id),d)
        topic_engine.mark_used(paths,topic)
        status.set_processor(paths,"qwen","idle",job_id,"ready")
        print("QWEN READY",job_id,topic.title)
    except Exception as e:
        d=read_json(paths.manifest(job_id),{}); d.update(status="QWEN_ERROR",error=str(e)); write_json_atomic(paths.manifest(job_id),d); status.set_processor(paths,"qwen","error",job_id,str(e)); print("ERROR",job_id,e)
